# Turkish Legal RAG - Hybrid Retrieval Grid Search

Bu notebook yeni model eğitmez. Mevcut BGE-M3 indexini kullanır; BGE-M3 modelini Kaggle working cache'e indirip local path üzerinden yükler.

Amaç mevcut `BGE-M3 + BM25 hybrid` sisteminin retrieval parametrelerini aramaktır:

- `alpha`: dense embedding skorunun ağırlığı
- `dense_candidates`: dense retrieval'dan alınan aday sayısı
- `bm25_candidates`: BM25'ten alınan aday sayısı
- `preliminary_top_k`: dedupe öncesi ara liste uzunluğu
- `metadata_profile`: soru içinde kanun/madde adı varsa küçük metadata boost kullanılıp kullanılmayacağı

Sonuçlar gold benchmark üzerinde ölçülür ve en iyi kombinasyon JSON/CSV olarak kaydedilir.

In [ ]:
!nvidia-smi

## 1. HuggingFace Ortam Ayarları

In [ ]:
import os

# Kaggle bazen HF/Xet download tarafında sessizce bekleyebiliyor.
# Bu ayar BGE-M3'ü normal HF download yolu ile almaya zorlar.
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("HF_HUB_DISABLE_XET =", os.environ["HF_HUB_DISABLE_XET"])

## 2. Paket Kontrolü

In [ ]:
# Kaggle ortamında genelde bu paketler hazır geliyor.
# Gereksiz pip upgrade ortamı bozabildiği için sadece eksikse kuruyoruz.
import importlib.util
import subprocess
import sys

required = {
    "sentence_transformers": "sentence-transformers",
    "faiss": "faiss-cpu",
    "huggingface_hub": "huggingface-hub",
    "tqdm": "tqdm",
}
missing = [pip_name for module_name, pip_name in required.items() if importlib.util.find_spec(module_name) is None]
print("Missing packages:", missing)

if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All required packages are already available.")

## 3. Dosyaları Working Klasörüne Kopyala

In [ ]:
from pathlib import Path
import shutil

INPUT_ROOT = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working/legal-rag")

for path in [
    WORK_DIR / "scripts",
    WORK_DIR / "data/processed",
    WORK_DIR / "data/index",
    WORK_DIR / "data/eval",
]:
    path.mkdir(parents=True, exist_ok=True)

def find_input_file(name: str) -> Path:
    matches = sorted(INPUT_ROOT.rglob(name))
    if not matches:
        raise FileNotFoundError(f"{name} not found under {INPUT_ROOT}. Kaggle dataset'e ekledin mi?")
    return matches[0]

def copy_required(name: str, dest_dir: Path) -> Path:
    src = find_input_file(name)
    dst = dest_dir / name
    shutil.copy2(src, dst)
    print(f"Copied {src} -> {dst}")
    return dst

for name in [
    "evaluate_retrieval.py",
    "grid_search_retrieval.py",
]:
    copy_required(name, WORK_DIR / "scripts")

for name in ["retrieval_chunks.json", "retrieval_corpus.json"]:
    copy_required(name, WORK_DIR / "data/processed")

for name in ["faiss_bge_m3.index", "metadata_bge_m3.json", "index_config_bge_m3.json"]:
    copy_required(name, WORK_DIR / "data/index")

copy_required("qa_benchmark_gold.csv", WORK_DIR / "data/eval")

print("\nWorking files:")
!find /kaggle/working/legal-rag -maxdepth 4 -type f | sort

## 4. BGE-M3 Modelini Local Cache'e Al ve Config'i Güncelle

In [ ]:
import json
from pathlib import Path
import torch
from huggingface_hub import snapshot_download
from sentence_transformers import SentenceTransformer

BGE_LOCAL_MODEL = Path("/kaggle/working/models/bge-m3")
BGE_LOCAL_MODEL.parent.mkdir(parents=True, exist_ok=True)

if not (BGE_LOCAL_MODEL / "modules.json").exists():
    print("Downloading BAAI/bge-m3 to local working cache...")
    snapshot_download(
        repo_id="BAAI/bge-m3",
        local_dir=str(BGE_LOCAL_MODEL),
        local_dir_use_symlinks=False,
        resume_download=True,
    )
else:
    print("Using existing local BGE-M3 cache:", BGE_LOCAL_MODEL)

config_path = WORK_DIR / "data/index/index_config_bge_m3.json"
config = json.loads(config_path.read_text(encoding="utf-8"))
config["original_model"] = config.get("model", "BAAI/bge-m3")
config["model"] = str(BGE_LOCAL_MODEL)
config_path.write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8")

print("Patched config model path:", config["model"])

print("GPU smoke test: loading local BGE-M3 on cuda...")
model = SentenceTransformer(str(BGE_LOCAL_MODEL), device="cuda")
emb = model.encode(
    ["query: işçi 2 gün işe gelmezse ne olur?"],
    batch_size=1,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)
print("Smoke test embedding shape:", emb.shape)

print("Precomputing dense FAISS scores once inside the notebook process...")
import faiss
import sys

scripts_dir = str(WORK_DIR / "scripts")
if scripts_dir not in sys.path:
    sys.path.insert(0, scripts_dir)

from evaluate_retrieval import build_corpus_lookups, load_benchmark

corpus_by_id, by_law_article, law_name_to_no = build_corpus_lookups(WORK_DIR / "data/processed/retrieval_corpus.json")
examples, coverage_stats = load_benchmark(
    WORK_DIR / "data/eval/qa_benchmark_gold.csv",
    by_law_article,
    law_name_to_no,
    active_only=True,
)
print("Resolved examples:", len(examples))
print("Coverage:", coverage_stats)

query_prefix = config.get("query_prefix", "query: ")
questions = [f"{query_prefix}{example['question']}" for example in examples]
embeddings = model.encode(
    questions,
    batch_size=16,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
).astype("float32")

index = faiss.read_index(str(WORK_DIR / "data/index/faiss_bge_m3.index"))
scores, indices = index.search(embeddings, 300)
dense_raw_list = []
for query_scores, query_indices in zip(scores, indices):
    dense_raw_list.append([
        [int(index_id), float(score)]
        for score, index_id in zip(query_scores, query_indices)
        if int(index_id) >= 0
    ])

dense_raw_path = WORK_DIR / "data/eval/dense_raw_bge_m3_top300.json"
dense_raw_path.write_text(
    json.dumps(
        {
            "embedding_model": str(BGE_LOCAL_MODEL),
            "top_k": 300,
            "dense_raw_list": dense_raw_list,
        },
        ensure_ascii=False,
    ),
    encoding="utf-8",
)
print("Saved precomputed dense scores to:", dense_raw_path)
del model
torch.cuda.empty_cache()

## 5. Mevcut Baseline'i Tekrar Ölç

In [ ]:
import json
from pathlib import Path

WORK_DIR = Path("/kaggle/working/legal-rag")

# Baseline daha önce aynı benchmark/index ile ölçüldü. Burada tekrar model
# yükletmiyoruz; grid içinde bu baseline kombinasyonu zaten tekrar denenecek.
baseline = {
    "summary": {
        "queries": 244,
        "recall@5": 0.6311475409836066,
        "recall@10": 0.6680327868852459,
        "top1_accuracy": 0.46311475409836067,
        "mrr": 0.5348214285714286,
        "ndcg@10": 0.5674053493059366,
    },
    "note": "Previously measured BGE-M3 + BM25 hybrid baseline on the same resolved gold benchmark.",
}
out = WORK_DIR / "data/eval/eval_hybrid_bge_m3_before_grid.json"
out.write_text(json.dumps(baseline, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(baseline["summary"], ensure_ascii=False, indent=2))

## 6. Grid Search Çalıştır

In [ ]:
import subprocess

cmd = [
    "python", "-u", str(WORK_DIR / "scripts/grid_search_retrieval.py"),
    "--benchmark", str(WORK_DIR / "data/eval/qa_benchmark_gold.csv"),
    "--corpus", str(WORK_DIR / "data/processed/retrieval_corpus.json"),
    "--chunks", str(WORK_DIR / "data/processed/retrieval_chunks.json"),
    "--index", str(WORK_DIR / "data/index/faiss_bge_m3.index"),
    "--metadata", str(WORK_DIR / "data/index/metadata_bge_m3.json"),
    "--config", str(WORK_DIR / "data/index/index_config_bge_m3.json"),
    "--embedding-device", "cuda",
    "--embedding-batch-size", "16",
    "--dense-raw-input", str(WORK_DIR / "data/eval/dense_raw_bge_m3_top300.json"),
    "--top-k", "10",
    "--alphas", "0.35,0.40,0.45,0.50,0.55,0.60,0.65,0.70",
    "--dense-candidates-list", "80,150,300",
    "--bm25-candidates-list", "100,250,500",
    "--preliminary-top-k-list", "50,80,120",
    "--metadata-profiles", "none,default,strong",
    "--output-json", str(WORK_DIR / "data/eval/grid_search_hybrid_bge_m3.json"),
    "--output-csv", str(WORK_DIR / "data/eval/grid_search_hybrid_bge_m3.csv"),
    "--top-n", "30",
]
subprocess.run(cmd, check=True)

## 7. En İyi Sonuçları Göster

In [ ]:
import json
import pandas as pd

baseline_path = WORK_DIR / "data/eval/eval_hybrid_bge_m3_before_grid.json"
grid_path = WORK_DIR / "data/eval/grid_search_hybrid_bge_m3.json"
csv_path = WORK_DIR / "data/eval/grid_search_hybrid_bge_m3.csv"

baseline = json.loads(baseline_path.read_text(encoding="utf-8"))
grid = json.loads(grid_path.read_text(encoding="utf-8"))

print("Baseline summary:")
print(json.dumps(baseline["summary"], ensure_ascii=False, indent=2))

print("\nBest grid result:")
print(json.dumps(grid["best"], ensure_ascii=False, indent=2))

df = pd.read_csv(csv_path)
display(df.head(20))

## 8. En İyi Parametrelerle Tekrar Eval Kaydet

In [ ]:
import json

grid = json.loads((WORK_DIR / "data/eval/grid_search_hybrid_bge_m3.json").read_text(encoding="utf-8"))
out = {
    "summary": grid["best"]["summary"],
    "params": grid["best"]["params"],
    "note": "This summary is produced by grid_search_retrieval.py using precomputed dense FAISS scores and BM25 scores.",
}
out_path = WORK_DIR / "data/eval/eval_hybrid_bge_m3_grid_best.json"
out_path.write_text(json.dumps(out, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(out, ensure_ascii=False, indent=2))

## 9. Kaydedilecek Dosyalar

In [ ]:
!ls -lh /kaggle/working/legal-rag/data/eval/*grid*
!ls -lh /kaggle/working/legal-rag/data/eval/eval_hybrid_bge_m3_before_grid.json

Önemli çıktı dosyaları:

```text
/kaggle/working/legal-rag/data/eval/grid_search_hybrid_bge_m3.json
/kaggle/working/legal-rag/data/eval/grid_search_hybrid_bge_m3.csv
/kaggle/working/legal-rag/data/eval/eval_hybrid_bge_m3_grid_best.json
```

Eğer en iyi grid sonucu baseline'dan iyi çıkarsa final retrieval parametrelerini bu değerlere güncelleriz. Eğer metadata boost ile iyileşme gelirse bunu ayrıca not etmek lazım, çünkü bu özellikle benchmark'ta kanun/madde adı geçen sorulara yardımcı olur.